The goal of this notebook is to take the large Transsee csv files and split it into managable chunks. We'll aim for csv files with
500k rows. This assumes that the large csv files

In [57]:
import pandas as pd
import numpy as np
import os

from torchgen.selective_build.operator import merge_debug_info

In [58]:
# Here I load the csv files in the current directory. Note that since the large files do not
# upload to github, running this notebook should do nothing.

csv_files = [file for file in os.listdir() if '.csv' == file[-4:]]
dataframe = [pd.read_csv(csv) for csv in csv_files]



/var/folders/6d/x65kjw5d1vl7fw7pnbx51yb80000gn/T/ipykernel_12364/2561852678.py:5: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = [pd.read_csv(csv) for csv in csv_files]


Sometimes it can happen that some of the csv files are missing columns. We will take the conservative approach and just put NaNs when there are columns missing.

In [59]:
combined_columns = set()
for df in dataframe:
    combined_columns = combined_columns.union(set(df.columns))

combined_columns = list(combined_columns)
print(combined_columns)

['Riders after stop', 'day of the week', 'Gap', 'Vehicle', 'Unnamed: 0', 'Schedule', 'Destination', 'day', 'Headway', 'Time']


In [60]:
# Add NaNs when a column is missing
for df in dataframe:
    for column in combined_columns:
        if column not in df.columns:
            df[column] = [np.nan for _ in range(len(df))]

merged_df = pd.concat(dataframe, ignore_index=True)

In [61]:
#Let's sort the merged_df by 'day'
merged_df.sort_values(by=['day'], inplace=True)

In [62]:
def split_dataframe_into_chunks(df: pd.DataFrame, chunk_size: int):
    return [df[k:k+chunk_size] for k in range(0, len(df), chunk_size)]

In [63]:
df_split = split_dataframe_into_chunks(merged_df, 500000)

In [64]:
data_name = '506_schedule_data'
for idx, df in enumerate(df_split):
    df.to_csv(f'raw_data/{data_name}_{idx}.csv', index=False)